#This tutorial requires following sample agent deployments
#https://github.com/cheeunlim/agent-engine-lab

In [ ]:
#AGENT_RUNTIME_RESOURCE_NAME follows below scheme
#projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{ENGINE_ID}
AGENT_RUNTIME_RESOURCE_NAME = f"projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{ENGINE_ID}"

In [2]:
from vertexai import Client, types
import render_util
import google.auth
from google.genai import types as genai_types
httpOptions = genai_types.HttpOptions(
    retry_options=genai_types.HttpRetryOptions(
        attempts=5,           # 최대 재시도 횟수
        initial_delay=1.0,    # 첫 대기 시간
        http_status_codes=[429, 500, 502, 503, 504] # 재시도 대상 에러 코드
    )
)
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
client = Client(project=PROJECT_ID, location=LOCATION)

In [3]:
#Prepare sample data
import pandas as pd
from vertexai._genai import types

session_inputs = types.evals.SessionInput(
    user_id="Test user",
    state={},
)
agent_prompts = [
    "치킨 커리 레시피가 궁금해요",
    "1주일 채식주의자를 위한 식단을 알려주세요",
]
agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})
agent_dataset

,prompt,session_inputs
0,치킨 커리 레시피가 궁금해요,user_id='Test user' state={} app_name=None
1,1주일 채식주의자를 위한 식단을 알려주세요,user_id='Test user' state={} app_name=None


In [4]:
#Get inference result
eval_dataset = client.evals.run_inference(
    agent=AGENT_RUNTIME_RESOURCE_NAME,
    src=agent_dataset,
)

render_util.display_evaluation_dataset(eval_dataset)

C:\Users\jeehyeok\AppData\Roaming\Python\Python312\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey
Agent Run: 100%|██████████| 2/2 [00:35<00:00, 17.50s/it]


In [5]:
eval_result = client.evals.evaluate(
        dataset=eval_dataset,
        metrics=[
            types.RubricMetric.GENERAL_QUALITY,
        ],
        config=types.EvaluateMethodConfig(
            http_options=httpOptions
        )
)
render_util.display_evaluation_result(eval_result)

Computing Metrics for Evaluation Dataset: 100%|██████████| 2/2 [00:27<00:00, 13.60s/it]
